<a href="https://colab.research.google.com/github/fzohra-ux/scalableML/blob/main/eval_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset
import json
import random

dataset = load_dataset("google-research-datasets/mbpp", "sanitized", split="test")

N=80
OUTPUT_FILE="mbpp_dataset.jsonl"
data=[dict(example) for example in dataset]

# # ---- Sample 500 random unique examples ----
sampled = random.sample(data, N)

# ---- Save to JSONL ----
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in sampled:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved dataset to {OUTPUT_FILE}")


README.md: 0.00B [00:00, ?B/s]

sanitized/train-00000-of-00001.parquet:   0%|          | 0.00/33.9k [00:00<?, ?B/s]

sanitized/test-00000-of-00001.parquet:   0%|          | 0.00/60.9k [00:00<?, ?B/s]

sanitized/validation-00000-of-00001.parq(…):   0%|          | 0.00/14.0k [00:00<?, ?B/s]

sanitized/prompt-00000-of-00001.parquet:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/257 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/43 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/7 [00:00<?, ? examples/s]

Saved dataset to mbpp_dataset.jsonl


In [ ]:
import json
import ast
from pathlib import Path
from difflib import SequenceMatcher


from gradio_client import Client
from tqdm import tqdm  # pip install tqdm


# ========= CONFIG =========
SPACE_ID = "FatimaZh/iris"  # <-- change this
API_NAME = "/chat"                       # <-- change if your endpoint is different

EVAL_FILE = "mbpp_dataset.jsonl"
RESULTS_FILE = "/content/drive/MyDrive/mbpp_eval_results.jsonl"


# ========= GRADIO CLIENT =========
client = Client(SPACE_ID)


Loaded as API: https://fatimazh-iris.hf.space ✔


In [ ]:
import ast


def build_prompt(example):
    """
    Build the evaluation prompt exactly like the training prompt.
    Your training format was:
        user: instruction (+ input)
        assistant: output
    So we mimic that here.
    """

    task = example["prompt"].strip()

    # MBPP is always a simple instruction-style natural language description.
    # Since CodeAlpaca uses "instruction [+ Input: ...]" format,
    # we format the MBPP task exactly the same way:
    user_message = task

    return f"Instruction: {user_message}\nAnswer:"






In [ ]:
import re

def get_expected_func_name_from_tests(tests_raw: str) -> str | None:
    """
    Parse the first function name used in the asserts, e.g.:
        'assert is_prime(2) == True; assert is_prime(5) == True'
    -> 'is_prime'
    """
    # Take the first assert statement
    for stmt in tests_raw:
        stmt = stmt.strip()
        if not stmt.startswith("assert"):
            continue

        # Regex: assert <func_name>(...
        m = re.search(r"assert\s+([a-zA-Z_]\w*)\s*\(", stmt)
        if m:
            return m.group(1)

    return None
def normalize_function_name(code_str: str, expected_name: str | None) -> str:
    """
    Replace the first function name in the generated code with `expected_name`.
    If `expected_name` is None or no function is found, return code_str unchanged.
    """
    if not expected_name:
        return code_str

    pattern = r"def\s+([a-zA-Z_]\w*)\s*\("
    m = re.search(pattern, code_str)
    if not m:
        return code_str  # no def found

    original_name = m.group(1)

    # Replace ONLY the definition name, not all occurrences
    return re.sub(
        rf"def\s+{original_name}\s*\(",
        f"def {expected_name}(",
        code_str,
        count=1,
    )


In [ ]:
def run_mbpp_tests(code_str: str, example) -> tuple[int, int]:
    """
    Run MBPP-style tests for a generated Python solution.
    Returns (passed, total).
    """
    tests_raw = (example.get("test_list") or "")
    if not tests_raw:
        return 0, 0

    setup_code = (
        example.get("test_imports")
        or ""
    )

    # 1) Find expected function name from asserts
    expected_name = get_expected_func_name_from_tests(tests_raw)

    # 2) Rename the first function in model output to that name
    code_str = normalize_function_name(code_str, expected_name)

    ns = {}

    try:
        # Setup (imports, etc.)
        if setup_code:
            exec(setup_code, ns, ns)

        # Execute model code (with normalized function name)
        exec(code_str, ns, ns)

    except Exception:
        # If it crashes before tests, all tests fail
        total = sum(
            1 for stmt in tests_raw
            if "assert" in stmt
        )
        return 0, total

    passed = 0
    total = 0

    for stmt in tests_raw:
        stmt = stmt.strip()
        if not stmt or not stmt.startswith("assert"):
            continue

        total += 1
        try:
            exec(stmt, ns, ns)
            passed += 1
        except Exception:
            # this assert failed
            pass

    return passed, total


In [ ]:
from pathlib import Path
import json
from tqdm import tqdm

# Make sure the eval file exists
eval_path = Path(EVAL_FILE)
assert eval_path.exists(), f"{EVAL_FILE} not found"

# Load all MBPP examples from the JSONL file
examples = []
with open(eval_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        examples.append(json.loads(line))

results = []

with open(RESULTS_FILE, "w", encoding="utf-8") as results_f:
    for ex in tqdm(examples, desc="Evaluating on MBPP-sanitized"):
        # ---- Build the prompt exactly like training (via build_prompt) ----
        prompt = build_prompt(ex)

        # ---- Call your model endpoint ----
        try:
            model_output = client.predict(
                prompt,
                api_name=API_NAME,
            )
            error = None
        except Exception as e:
            model_output = ""
            error = str(e)

        # ---- Run functional tests (MBPP asserts) ----
        if error is None:
            passed_tests, total_tests = run_mbpp_tests(str(model_output), ex)
        else:
            # If the model call failed, count all tests as failed
            tests_raw = (ex.get("test_list") or "")
            total_tests = sum(
                1 for stmt in tests_raw
                if stmt.strip().startswith("assert")
            )
            passed_tests = 0

        out_row = {
            "task_id": ex.get("task_id", None),
            "prompt": ex.get("prompt", ""),
            "reference_code": ex.get("code", ""),
            "model_output": model_output,
            "tests_passed": passed_tests,
            "tests_total": total_tests,
            "error": error,
        }

        results.append(out_row)
        results_f.write(json.dumps(out_row, ensure_ascii=False) + "\n")

# ========= SUMMARY STATS =========
print("==== EVAL SUMMARY (MBPP-sanitized) ====")

# Per-assert pass rate
all_passed = sum(r["tests_passed"] for r in results)
all_total = sum(r["tests_total"] for r in results)
if all_total > 0:
    print(f"Overall assert pass rate: {all_passed}/{all_total} "
          f"({all_passed / all_total * 100:.2f}%)")
else:
    print("No tests available in these MBPP examples.")

# Per-task success: all tests for that task passed
tasks_with_tests = [r for r in results if r["tests_total"] > 0]
if tasks_with_tests:
    fully_solved = sum(
        1 for r in tasks_with_tests
        if r["tests_passed"] == r["tests_total"]
    )
    print(f"Tasks fully solved: {fully_solved}/{len(tasks_with_tests)} "
          f"({fully_solved / len(tasks_with_tests) * 100:.2f}%)")
else:
    print("No tasks had any tests.")

errors = [r for r in results if r["error"] is not None]
if errors:
    print(f"Model call errors on {len(errors)} / {len(results)} examples.")

print(f"Detailed per-example results written to: {RESULTS_FILE}")


Evaluating on MBPP-sanitized: 100%|██████████| 80/80 [32:40<00:00, 24.51s/it]

==== EVAL SUMMARY (MBPP-sanitized) ====
Overall assert pass rate: 86/240 (35.83%)
Tasks fully solved: 25/80 (31.25%)
Detailed per-example results written to: /content/drive/MyDrive/mbpp_eval_results.jsonl


In [ ]:
client.view_api(SPACE_ID)

Client.predict() Usage Info
---------------------------
Named API endpoints: 1

 - predict(message, system_message, max_tokens, temperature, top_p, api_name="/chat") -> response
    Parameters:
     - [Textbox] message: str (required)  
     - [Textbox] system_message: str (not required, defaults to:   You are a helpful and friendly assistant.)  
     - [Slider] max_tokens: float (not required, defaults to:   256)  
     - [Slider] temperature: float (not required, defaults to:   0.8)  
     - [Slider] top_p: float (not required, defaults to:   0.9)  
    Returns:
     - [Json] response: str | float | bool | list | dict (any valid json) 

Unnamed API endpoints: 0

